# 04 - Locked SE-ResNeXt-50 YOLO ROI Test Evaluation

Run after notebook 03. This evaluates its selected checkpoint exactly once on
the locked YOLO ROI test split. It creates metrics, a confusion matrix, and
native-CAM examples for correct and misclassified cases. Do not use these test
results to choose a different checkpoint or training configuration.


In [ ]:
!pip -q install "timm>=1.0"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## Fixed evaluation configuration

The checkpoint path is the selected epoch from Notebook 03. Keep the test split locked: do not use these metrics to choose another epoch or preprocessing setting.

In [ ]:
from __future__ import annotations

import json
import random
from datetime import datetime, timezone
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
import torchvision.transforms as transforms
from sklearn.metrics import (
    accuracy_score, average_precision_score, classification_report,
    cohen_kappa_score, confusion_matrix, precision_recall_fscore_support,
    recall_score, roc_auc_score,
)
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

SEED = 42
INPUT_SIZE = 384
BATCH_SIZE = 48
NUM_WORKERS = 2
CASES_PER_GRADE = 5

ROI_TEST_ROOT = Path(
    "/content/drive/MyDrive/Datasets/KneeXrayData_Mendeley_v1/derived/"
    "densenet121_yolo_square_roi_trainvaltest_v2/test"
)
CHECKPOINT_PATH = Path(
    "/content/drive/MyDrive/Models/seresnext50_32x4d_paired_view_adaptation/"
    "SET_THIS_TO_NOTEBOOK_03_RUN/best_model.pth"
)
RUN_TIMESTAMP = datetime.now(timezone.utc).strftime("%Y-%m-%d_%H-%M-%S_%f_UTC")
OUTPUT_DIR = (
    Path("/content/drive/MyDrive/Models/seresnext50_32x4d_paired_view_evaluation")
    / RUN_TIMESTAMP
)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

for required in (ROI_TEST_ROOT, CHECKPOINT_PATH):
    if not required.exists():
        raise FileNotFoundError(f"Required path not found: {required}")

OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("Device:", DEVICE)
print("Checkpoint:", CHECKPOINT_PATH)
print("Locked ROI test split:", ROI_TEST_ROOT)
print("Output directory:", OUTPUT_DIR)

## Reproduce the Notebook 03 evaluation transform

This is the deterministic path used for ROI validation during paired-view fine-tuning: CLAHE at clip limit 1.25, square padding, resize to 384, and normalization. It deliberately contains no random augmentation and no centre crop.

In [ ]:
class OpenCVCLAHE:
    def __call__(self, image_rgb: np.ndarray) -> np.ndarray:
        lab = cv2.cvtColor(np.asarray(image_rgb), cv2.COLOR_RGB2LAB)
        lightness, channel_a, channel_b = cv2.split(lab)
        lightness = cv2.createCLAHE(clipLimit=1.25, tileGridSize=(8, 8)).apply(lightness)
        return cv2.cvtColor(
            cv2.merge((lightness, channel_a, channel_b)), cv2.COLOR_LAB2RGB
        )


class SquarePad:
    def __call__(self, image_rgb: np.ndarray) -> np.ndarray:
        image = np.asarray(image_rgb)
        height, width = image.shape[:2]
        side = max(height, width)
        top = (side - height) // 2
        left = (side - width) // 2
        return cv2.copyMakeBorder(
            image, top, side - height - top, left, side - width - left,
            cv2.BORDER_CONSTANT, value=(0, 0, 0),
        )


display_transform = transforms.Compose([
    OpenCVCLAHE(),
    SquarePad(),
    transforms.ToPILImage(),
    transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
])
tensor_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])


def prepare_roi(path: str | Path) -> tuple[np.ndarray, torch.Tensor]:
    image_bgr = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if image_bgr is None:
        raise RuntimeError(f"Cannot decode ROI: {path}")
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    processed_rgb = np.array(display_transform(image_rgb), dtype=np.uint8, copy=True)
    return processed_rgb, tensor_transform(processed_rgb)


class ROITestDataset(Dataset):
    def __init__(self, root: Path):
        rows = []
        for grade in range(5):
            paths = sorted((root / str(grade)).glob("*.png"))
            if not paths:
                raise RuntimeError(f"No ROI PNGs found for grade {grade}: {root / str(grade)}")
            rows.extend({"path": str(path), "true_grade": grade} for path in paths)
        self.frame = pd.DataFrame(rows)

    def __len__(self) -> int:
        return len(self.frame)

    def __getitem__(self, index: int):
        row = self.frame.iloc[index]
        _, tensor = prepare_roi(row.path)
        return tensor, int(row.true_grade), row.path


test_dataset = ROITestDataset(ROI_TEST_ROOT)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
)
print(test_dataset.frame.groupby("true_grade").size())
print(f"Total ROIs: {len(test_dataset)}")

## Load checkpoint and evaluate once

The test split is used only for final reporting. This cell writes machine-readable predictions and metrics, then displays the confusion matrix.

In [ ]:
class SEResNeXt50NativeCAM(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = timm.create_model("seresnext50_32x4d", pretrained=False, features_only=True, out_indices=(4,))
        self.class_conv = nn.Conv2d(self.backbone.feature_info.channels()[0], 5, kernel_size=1)

    def class_maps(self, images):
        return self.class_conv(self.backbone(images)[0])

    def forward(self, images):
        return self.class_maps(images).mean(dim=(2, 3))

    @torch.no_grad()
    def native_cam(self, images, classes):
        maps = self.class_maps(images)
        selected = F.relu(maps[torch.arange(images.size(0), device=images.device), classes])
        selected = F.interpolate(selected[:, None], size=images.shape[-2:], mode="bilinear", align_corners=False)[:, 0]
        return selected / selected.flatten(1).amax(1).clamp_min(1e-8)[:, None, None]


checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)
if checkpoint.get("loss_type") not in (None, "ce"):
    raise RuntimeError(f"Expected a CE checkpoint, got {checkpoint.get('loss_type')}")
if checkpoint.get("architecture") not in ("final_native_cam_ce", "natural_final_native_cam_ce"):
    raise RuntimeError(f"Unexpected architecture: {checkpoint.get('architecture')}")
model = SEResNeXt50NativeCAM().to(DEVICE)
model.load_state_dict(checkpoint["model_state_dict"], strict=True)
model.eval()

all_paths, all_labels, all_probabilities = [], [], []
with torch.inference_mode():
    for images, labels, paths in tqdm(test_loader, desc="Locked ROI test evaluation"):
        probabilities = F.softmax(model(images.to(DEVICE, non_blocking=True)).float(), dim=1)
        all_paths.extend(paths); all_labels.extend(labels.numpy().tolist()); all_probabilities.extend(probabilities.cpu().numpy())
labels, probabilities = np.asarray(all_labels, dtype=int), np.asarray(all_probabilities, dtype=float)
predictions = probabilities.argmax(axis=1)
precision, recall, macro_f1, _ = precision_recall_fscore_support(labels, predictions, average="macro", zero_division=0)
metrics = {"samples": int(len(labels)), "accuracy": float(accuracy_score(labels, predictions)), "qwk": float(cohen_kappa_score(labels, predictions, weights="quadratic")), "macro_precision": float(precision), "macro_recall": float(recall), "macro_f1": float(macro_f1), "macro_ap": float(average_precision_score(np.eye(5)[labels], probabilities, average="macro")), "macro_roc_auc_ovr": float(roc_auc_score(labels, probabilities, multi_class="ovr", average="macro") )}
metrics["selection_reference_only"] = float(0.55 * metrics["qwk"] + 0.30 * metrics["macro_f1"] + 0.15 * metrics["macro_ap"])
prediction_frame = pd.DataFrame({"roi_path": all_paths, "true_grade": labels, "predicted_grade": predictions, "confidence": probabilities.max(axis=1)})
for grade in range(5): prediction_frame[f"probability_grade_{grade}"] = probabilities[:, grade]
prediction_frame.to_csv(OUTPUT_DIR / "test_predictions.csv", index=False)
(OUTPUT_DIR / "test_metrics.json").write_text(json.dumps(metrics, indent=2))
print(json.dumps(metrics, indent=2))
figure, axis = plt.subplots(figsize=(7, 6))
sns.heatmap(confusion_matrix(labels, predictions, labels=range(5)), annot=True, fmt="d", cmap="Blues", xticklabels=range(5), yticklabels=range(5), ax=axis)
axis.set(xlabel="Predicted KL grade", ylabel="True KL grade", title="Locked YOLO ROI Test Confusion Matrix")
figure.tight_layout(); figure.savefig(OUTPUT_DIR / "test_confusion_matrix.png", dpi=180, bbox_inches="tight"); plt.show()


## Grad-CAM review

Correct cases show the evidence used when the prediction is right. Incorrect cases create two CAMs for each image: one for the predicted class and one for the true class. This helps distinguish anatomically plausible class confusion from shortcut evidence outside the joint.

In [ ]:
def overlay(image, cam):
    heat = cv2.cvtColor(cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET), cv2.COLOR_BGR2RGB)
    return cv2.addWeighted(image, 0.58, heat, 0.42, 0)


def render(path, class_index):
    image, tensor = prepare_roi(path)
    cam = model.native_cam(tensor[None].to(DEVICE), torch.tensor([class_index], device=DEVICE))[0].cpu().numpy()
    return image, overlay(image, cam)


correct_dir = OUTPUT_DIR / "native_cam_correct_by_true_grade"; correct_dir.mkdir()
incorrect_dir = OUTPUT_DIR / "native_cam_misclassified_pairs"; incorrect_dir.mkdir()
rows = []
for grade in range(5):
    good = prediction_frame[(prediction_frame.true_grade == grade) & (prediction_frame.predicted_grade == grade)].head(CASES_PER_GRADE)
    figure, axes = plt.subplots(2, CASES_PER_GRADE, figsize=(4 * CASES_PER_GRADE, 7))
    for column in range(CASES_PER_GRADE):
        axes[0, column].axis("off"); axes[1, column].axis("off")
        if column < len(good):
            case = good.iloc[column]; image, cam = render(case.roi_path, grade)
            axes[0, column].imshow(image); axes[0, column].set_title(f"True G{grade}")
            axes[1, column].imshow(cam); axes[1, column].set_title(f"Native CAM p={case.confidence:.3f}")
    figure.tight_layout(); figure.savefig(correct_dir / f"grade_{grade}_correct.png", dpi=160, bbox_inches="tight"); plt.show(); plt.close(figure)
    bad = prediction_frame[(prediction_frame.true_grade == grade) & (prediction_frame.predicted_grade != grade)].head(CASES_PER_GRADE)
    for number, (_, case) in enumerate(bad.iterrows(), 1):
        image, pred_cam = render(case.roi_path, int(case.predicted_grade)); _, true_cam = render(case.roi_path, int(case.true_grade))
        figure, axes = plt.subplots(1, 3, figsize=(14, 5))
        axes[0].imshow(image); axes[0].set_title(f"ROI | true G{case.true_grade}")
        axes[1].imshow(pred_cam); axes[1].set_title(f"Predicted G{case.predicted_grade}")
        axes[2].imshow(true_cam); axes[2].set_title(f"True-class native CAM")
        for axis in axes: axis.axis("off")
        output = incorrect_dir / f"true_g{case.true_grade}_pred_g{case.predicted_grade}_{number}_{Path(case.roi_path).stem}.png"
        figure.tight_layout(); figure.savefig(output, dpi=160, bbox_inches="tight"); plt.show(); plt.close(figure)
        rows.append({"roi_path": case.roi_path, "true_grade": int(case.true_grade), "predicted_grade": int(case.predicted_grade), "confidence": float(case.confidence), "cam_pair_path": str(output)})
pd.DataFrame(rows).to_csv(OUTPUT_DIR / "native_cam_misclassified_index.csv", index=False)


## Save evaluation manifest

Use the output directory as the evidence package for the report. Promotion to the application requires both acceptable test metrics and a manual review showing joint-space-focused Grad-CAMs without systematic border, marker, femur, or tibia shortcuts.

In [ ]:
manifest = {"checkpoint_path": str(CHECKPOINT_PATH), "test_roi_root": str(ROI_TEST_ROOT), "preprocessing": "CLAHE(1.25) -> square pad -> resize(384) -> ImageNet normalization", "architecture": "final_native_cam_ce", "evaluation_only": True, "metrics": metrics, "correct_native_cam_directory": str(correct_dir), "misclassified_native_cam_directory": str(incorrect_dir)}
(OUTPUT_DIR / "evaluation_manifest.json").write_text(json.dumps(manifest, indent=2))
print("Evaluation complete. Evidence package:", OUTPUT_DIR)
